# Chopp & Cia · 02 — Ingestão

**Projeto Integrador VI** · FATEC Votorantim · 2º Semestre/2026

Publica o CSV consolidado como **tabela gerenciada** no Unity Catalog.

| | |
|:---|:---|
| **Entrada** | `dataset_consolidado_v<DATA_VERSION>.csv` num Volume |
| **Saída** | `<catálogo>.<schema>.dataset_consolidado_v<DATA_VERSION>` |
| **Ambiente** | Databricks · requer Spark |
| **Anterior** | 01 produz o CSV, em Windows local |

Este notebook **não transforma dado**. Ele valida o contrato de colunas e grava.
Toda regra de negócio vive no notebook 01 — duas implementações da mesma regra
divergem no dia em que alguém corrige só uma.

> **Só roda no Databricks.** Sem Spark não há tabela para publicar; localmente,
> os notebooks 03 a 07 leem o CSV do 01 diretamente.

## 1. Painel de controle

Preencha os quatro campos abaixo. `DATA_VERSION` deve ser a mesma do CSV que o
notebook 01 produziu — é ela que nomeia a tabela e amarra o resultado aos dados
que o geraram.

In [ ]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd

CAMINHO_CSV = "/Volumes/projetointegrador/projetointegrador/dados/dataset_consolidado_v1_0.csv"
CATALOGO = "projetointegrador"
SCHEMA = "projetointegrador"
DATA_VERSION = "1.0"

SOBRESCREVER = False       # protege uma tabela já publicada desta versão

try:
    spark                                                  # noqa: F821
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False

if not EM_DATABRICKS:
    raise RuntimeError(
        "Este notebook publica no Unity Catalog e só roda no Databricks.\n"
        "Para trabalhar localmente, use o CSV do notebook 01 direto nos 03 a 07."
    )

if not re.fullmatch(r"\d+\.\d+", DATA_VERSION):
    raise ValueError("DATA_VERSION deve seguir o formato MAIOR.MENOR, por exemplo 1.0.")

SUFIXO = DATA_VERSION.replace(".", "_")
TABELA_DESTINO = f"{CATALOGO}.{SCHEMA}.dataset_consolidado_v{SUFIXO}"
VIEW_CORRENTE = f"{CATALOGO}.{SCHEMA}.dataset_consolidado_corrente"

print(f"Entrada : {CAMINHO_CSV}")
print(f"Destino : {TABELA_DESTINO}")

## 2. Guarda de versão

Falha **antes** de ler o CSV. Publicar sobre uma versão existente faz os
resultados que apontam para ela deixarem de ser reproduzíveis, e nada indicaria
isso depois.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA}")            # noqa: F821

if spark.catalog.tableExists(TABELA_DESTINO) and not SOBRESCREVER:       # noqa: F821
    raise RuntimeError(
        f"A tabela {TABELA_DESTINO} já existe.\n"
        f"Incremente DATA_VERSION no painel, ou use SOBRESCREVER = True sabendo "
        f"que os resultados que citam a versão {DATA_VERSION} deixarão de bater."
    )

print(f"Versão {DATA_VERSION} disponível para publicação.")

## 3. Carga e contrato

As 38 colunas abaixo são o que o notebook 01 promete entregar e o que os
notebooks 03 a 07 esperam encontrar. Coluna faltando aborta aqui, e não trinta
células adiante com um `KeyError` sem contexto.

A guarda de identificação nominal se repete aqui de propósito: se uma carga
futura reintroduzir uma coluna de nome, ela para nesta célula em vez de entrar
em silêncio na tabela.

In [ ]:
CONTRATO = [
    "ID_PESSOA", "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO", "MODALIDADE", "TEM_COMISSAO",
    "PRIMEIRA_COMPRA", "ULTIMA_COMPRA", "MES_ULTIMA_COMPRA",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO",
    "TICKET_MEDIO", "PRODUTO_FAVORITO", "PCT_COMPRAS_A_PRAZO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
    "AGING_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
    "AGING_COMODATO", "PRAZO_MEDIO_COMODATO", "VALOR_MEDIO_COMODATO",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
    "_DATA_VERSION",
]

TEXTUAIS = {
    "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO", "MODALIDADE", "MES_ULTIMA_COMPRA",
    "PRODUTO_FAVORITO", "AGING_PAGAMENTO", "AGING_COMODATO", "PERFIL_RISCO",
    "_DATA_VERSION",
}
DATAS = {"PRIMEIRA_COMPRA", "ULTIMA_COMPRA"}

caminho = Path(CAMINHO_CSV.strip())
if not caminho.exists():
    raise FileNotFoundError(f"CSV não encontrado: {caminho}")

try:
    dataset = pd.read_csv(caminho, sep=";", encoding="utf-8-sig", low_memory=False)
except UnicodeDecodeError:
    dataset = pd.read_csv(caminho, sep=";", encoding="latin-1", low_memory=False)

dataset.columns = [str(c).strip().upper() for c in dataset.columns]

faltantes = [c for c in CONTRATO if c not in dataset.columns]
if faltantes:
    raise ValueError(f"CSV fora do contrato. Colunas ausentes: {faltantes}")

PADROES_NOMINAIS = ("NOME", "NM_", "FANTASIA", "RAZAO", "CPF", "CNPJ", "EMAIL",
                    "TELEFONE", "ENDERECO")
nominais = [c for c in dataset.columns if any(p in c for p in PADROES_NOMINAIS)]
if nominais:
    raise ValueError(f"Colunas de identificação nominal não permitidas: {nominais}")

if dataset.empty:
    raise ValueError("O CSV consolidado está vazio.")
if dataset["ID_PESSOA"].isna().any():
    raise ValueError("ID_PESSOA contém valores vazios.")
if dataset["ID_PESSOA"].duplicated().any():
    raise ValueError(
        f"{int(dataset['ID_PESSOA'].duplicated().sum())} ID_PESSOA duplicado(s): "
        "o consolidado deve ter uma linha por cliente."
    )

# Colunas fora do contrato são descartadas: a tabela publicada é exatamente o
# que os notebooks seguintes esperam, nem mais nem menos.
dataset = dataset[CONTRATO]

for coluna in DATAS:
    dataset[coluna] = pd.to_datetime(dataset[coluna], errors="coerce")
for coluna in CONTRATO:
    if coluna not in TEXTUAIS and coluna not in DATAS:
        dataset[coluna] = pd.to_numeric(dataset[coluna], errors="coerce")

print(f"CSV validado: {len(dataset):,} clientes × {dataset.shape[1]} colunas")

## 4. Auditoria

Confere que os números do CSV batem com o que o notebook 01 reportou. Divergência
aqui significa CSV trocado ou corrompido no caminho até o Volume.

In [ ]:
print(f"{'clientes únicos':<24}{dataset['ID_PESSOA'].nunique():>8,}")
for flag, rotulo in [("TEM_VENDAS", "com vendas"), ("TEM_FINANCEIRO", "com financeiro"),
                     ("TEM_COMODATO", "com comodato"), ("CORE_BUSINESS", "core business")]:
    n = int(dataset[flag].fillna(0).sum())
    print(f"{rotulo:<24}{n:>8,} ({n / len(dataset) * 100:>5.1f}%)")

print(f"\n{'faturamento total':<24}R$ {dataset['TOTAL_GASTO'].sum():>15,.2f}")
print(f"{'clientes em risco':<24}{int((dataset['PERFIL_RISCO'] != 'SEM RISCO').sum()):>8,}")

## 5. Publicação

Grava com `saveAsTable()` — **tabela do catálogo**, não arquivos soltos num
Volume. A diferença é prática: a tabela aparece no Catalog Explorer e no
`SHOW TABLES`, é consultável por SQL, aceita `GRANT` por objeto e carrega o
dicionário de dados nos comentários de coluna.

A view `_corrente` é conveniência para consulta ad-hoc. **Não use em
experimento**: ela muda quando uma versão nova é publicada, e um experimento
apontado para ela deixa de ser reproduzível.

In [ ]:
DICIONARIO = {
    "ID_PESSOA": "Chave do cliente — unidade de análise de todo o projeto",
    "PERFIL": "Física | Jurídica | Estrangeiro",
    "CIDADE": "Município padronizado (agrupa variações de digitação)",
    "PAGAMENTO": "Forma de pagamento padrão do cadastro",
    "SEGMENTO": "Tipo de estabelecimento (cadastro; sabidamente ruidoso)",
    "MODALIDADE": "Modalidade de atendimento do cliente (cadastro)",
    "TEM_COMISSAO": "1 se a venda ao cliente gera comissão (cadastro)",
    "PRIMEIRA_COMPRA": "Data do primeiro pedido; nulo se nunca comprou",
    "ULTIMA_COMPRA": "Data do último pedido; nulo se nunca comprou",
    "MES_ULTIMA_COMPRA": "Período AAAA-MM da última compra",
    "DIAS_DESDE_PRIMEIRA_COMPRA": "Dias entre a primeira compra e a data de corte",
    "DIAS_DESDE_ULTIMA_COMPRA": "Dias desde a última compra (R do RFM)",
    "FREQUENCIA_COMPRAS": "Pedidos distintos (F do RFM)",
    "TOTAL_ITENS": "Linhas de item de pedido",
    "QTD_TOTAL_VENDIDA": "Unidades vendidas somadas",
    "TOTAL_GASTO": "Receita total do cliente (M do RFM)",
    "TICKET_MEDIO": "TOTAL_GASTO / FREQUENCIA_COMPRAS",
    "PRODUTO_FAVORITO": "Produto mais consumido em volume",
    "PCT_COMPRAS_A_PRAZO": "Fração das parcelas contratadas como venda a prazo (vs. à vista)",
    "TOTAL_PARCELAS": "Parcelas de contas a receber",
    "PARCELAS_ATRASADAS": "Parcelas pagas em atraso ou vencidas e não pagas",
    "TAXA_ATRASO_PAGAMENTO": "PARCELAS_ATRASADAS / TOTAL_PARCELAS — base do alvo",
    "MEDIA_DIAS_ATRASO_PAG": "Média de dias de atraso no pagamento",
    "MAX_DIAS_ATRASO_PAG": "Pior atraso de pagamento observado",
    "VALOR_TOTAL_PARCELAS": "Soma do valor das parcelas",
    "AGING_PAGAMENTO": "Faixa de severidade do PIOR atraso de pagamento",
    "TOTAL_COMODATOS": "Contratos de comodato do cliente",
    "COMODATOS_ATRASADOS": "Contratos devolvidos em atraso ou vencidos e não devolvidos",
    "TAXA_ATRASO_COMODATO": "COMODATOS_ATRASADOS / TOTAL_COMODATOS — base do alvo",
    "MEDIA_DIAS_ATRASO_COM": "Média de dias de atraso na devolução",
    "MAX_DIAS_ATRASO_COM": "Pior atraso de devolução observado",
    "QTD_EQUIPAMENTOS": "Unidades de equipamento cedidas",
    "AGING_COMODATO": "Faixa de severidade do PIOR atraso de devolução",
    "PRAZO_MEDIO_COMODATO": "Prazo médio ACORDADO nos contratos de comodato (dias)",
    "VALOR_MEDIO_COMODATO": "Valor médio ACORDADO nos contratos de comodato (R$)",
    "RISCO_FINANCEIRO": "1 se a taxa de atraso de pagamento passa do limite da EDA",
    "RISCO_COMODATO": "1 se a taxa de atraso de comodato passa do limite da EDA",
    "PERFIL_RISCO": "RISCO DUPLO | SÓ FINANCEIRO | SÓ COMODATO | SEM RISCO",
    "CORE_BUSINESS": "1 se o cliente comprou chopp/chopeira — universo de modelagem",
    "TEM_VENDAS": "1 se o cliente tem histórico de vendas",
    "TEM_FINANCEIRO": "1 se o cliente tem parcelas registradas",
    "TEM_COMODATO": "1 se o cliente tem contratos de comodato",
    "_DATA_VERSION": "Versão da carga que produziu esta linha",
}

# Coluna 'object' só com nulos vira NullType no Spark e quebra a escrita.
pronto = dataset.copy()
for coluna in pronto.select_dtypes(include="object").columns:
    pronto[coluna] = pronto[coluna].fillna("").astype(str)

(
    spark.createDataFrame(pronto).write                       # noqa: F821
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)
print(f"Tabela gravada: {len(dataset):,} linhas × {dataset.shape[1]} colunas")

spark.sql(f"""
    ALTER TABLE {TABELA_DESTINO} SET TBLPROPERTIES (
        'data_version' = '{DATA_VERSION}',
        'origem_csv'   = '{caminho.name}',
        'carga_ts'     = '{datetime.now().isoformat(timespec="seconds")}'
    )
""")

aplicados = 0
for coluna, descricao in DICIONARIO.items():
    try:
        spark.sql(                                            # noqa: F821
            f"ALTER TABLE {TABELA_DESTINO} ALTER COLUMN {coluna} "
            f"COMMENT '{descricao.replace(chr(39), chr(39) * 2)}'"
        )
        aplicados += 1
    except Exception as erro:
        print(f"  comentário de {coluna} não aplicado: {type(erro).__name__}")
print(f"{aplicados} comentários de coluna aplicados")

spark.sql(f"""
    CREATE OR REPLACE VIEW {VIEW_CORRENTE}
    COMMENT 'Aponta para a carga mais recente ({TABELA_DESTINO}).
             Para experimentos use a TABELA versionada, não esta view.'
    AS SELECT * FROM {TABELA_DESTINO}
""")

print(f"\nTabela  : {TABELA_DESTINO}")
print(f"View    : {VIEW_CORRENTE}")
print("\nPróximo passo: nos notebooks 03 a 07, aponte a tabela de entrada para")
print(f"  {TABELA_DESTINO}")